# COMP 9130 — Capstone Project: Military Camouflage Object Detection
## Notebook 02: Training — Experiment 1 (SINetV2 on ACD1K)

**Author:** Yansong Jia (Experiment 1 Lead)  
**Dataset:** ACD1K only (military camouflage soldiers)  
**Model:** SINetV2 (lightweight encoder–decoder)

**Purpose:** Train a SINetV2 baseline on the ACD1K dataset at 352×352
resolution and export the best validation-checkpoint and training history
for unified final evaluation.

---

### Experiment 1 Design

| Setting | Value |
|---|---|
| Model | SINetV2 (custom PyTorch, binary output) |
| Training data | ACD1K train (748 images) |
| Validation data | ACD1K val (230 images, fixed splits) |
| Input resolution | **352 × 352** |
| Optimizer | AdamW |
| Loss | Binary cross-entropy + Dice |
| Early stopping | patience = 10 epochs (val mIoU) |
| LR sweep | 1e-4, 6e-5, 1e-5 |
| Outputs | `outputs/exp1/final/best_model.pth`, `history.json` |

This notebook deliberately **does not** run any final 200-image hold-out
evaluation or cross-experiment comparison. Those steps are handled by
Sepehr's final evaluation notebook.

---
## 1. Environment Setup

In [ ]:
# Cell 1 — Clone repo
!git clone https://github.com/bing-er/AI-final-project.git
%cd /content/AI-final-project

In [ ]:
# Cell 2 — Install dependencies
!pip install -q -r requirements.txt kaggle

In [ ]:
# (Optional) Cell 3 — Mount Google Drive to save outputs
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

---
## 2. Dataset Download (ACD1K only)

Experiment 1 uses only **ACD1K**. COD10K and CAMO are not required here.

In [ ]:
# Cell 4 — Upload kaggle.json (run once per session)
import os
from google.colab import files

if not os.path.exists('/root/.config/kaggle/kaggle.json'):
    files.upload()  # select kaggle.json from your machine when prompted
    os.makedirs('/root/.config/kaggle', exist_ok=True)
    os.rename('kaggle.json', '/root/.config/kaggle/kaggle.json')
    os.chmod('/root/.config/kaggle/kaggle.json', 0o600)
    print('Kaggle credentials configured')
else:
    print('Kaggle credentials already configured')

In [ ]:
# Cell 5 — Download ACD1K (~350 MB)
import os

acd_train = 'data/dataset-splitM/Training/images'
if not os.path.exists(acd_train):
    !kaggle datasets download \
        -d aalihhiader/military-camouflage-soldiers-dataset-mcs1k \
        -p data/ --unzip
else:
    print('✅ ACD1K already exists, skipping download')

---
## 3. Dataset Verification (ACD1K paths only)

In [ ]:
# Cell 6 — Verify ACD1K folder structure
import os

expected = [
    'data/dataset-splitM/Training/images',
    'data/dataset-splitM/Training/GT',
    'data/dataset-splitM/Testing/images',
    'data/dataset-splitM/Testing/GT',
]
for p in expected:
    status = '✅' if os.path.exists(p) else '❌ NOT FOUND'
    print(f'{status} — {p}')

In [ ]:
# Cell 7 — Quick ACD1K-only sanity check
import torch
from pathlib import Path

import sys
sys.path.insert(0, str(Path.cwd() / 'src'))
from dataset import build_dataloader

train_loader = build_dataloader(
    'data/', condition='acd1k', split='train',
    batch_size=4, num_workers=2,
    oversample_acd1k=False, seed=42,
    splits_dir='splits/',
)
val_loader = build_dataloader(
    'data/', condition='acd1k', split='val',
    batch_size=4, num_workers=2,
    oversample_acd1k=False, seed=42,
    splits_dir='splits/',
)

batch = next(iter(train_loader))
print('Train batch image shape:', batch['image'].shape)
print('Train batch mask  shape:', batch['mask'].shape)
print('Train datasets:', set(batch['dataset']))

batch_val = next(iter(val_loader))
print('Val batch image shape  :', batch_val['image'].shape)
print('Val batch mask  shape  :', batch_val['mask'].shape)
print('Val datasets:', set(batch_val['dataset']))

---
## 4. GPU Verification

In [ ]:
# Cell 8 — Check GPU
!nvidia-smi

---
## 5. Learning Rate Sweep (val mIoU on ACD1K)

We run three short training runs with different learning rates on ACD1K,
using validation mIoU to select the best LR for the final run.

- Run 1: `lr=1e-4`  
- Run 2: `lr=6e-5`  
- Run 3: `lr=1e-5`  

All runs write their history and (temporary) best_model checkpoints under
`outputs/exp1/sweep_lr*/`. Only the **final** run below writes to
`outputs/exp1/final/`.

In [ ]:
# Cell 9 — Sweep run 1 (lr=1e-4)
!PYTHONPATH=/content/AI-final-project/src \
 python src/engine_exp1.py \
    --lr 1e-4 --epochs 30 --batch_size 8 \
    --data_root data/ --splits_dir splits/ \
    --output_dir outputs/exp1/sweep_lr1e4

In [ ]:
# Cell 10 — Sweep run 2 (lr=6e-5)
!PYTHONPATH=/content/AI-final-project/src \
 python src/engine_exp1.py \
    --lr 6e-5 --epochs 30 --batch_size 8 \
    --data_root data/ --splits_dir splits/ \
    --output_dir outputs/exp1/sweep_lr6e5

In [ ]:
# Cell 11 — Sweep run 3 (lr=1e-5)
!PYTHONPATH=/content/AI-final-project/src \
 python src/engine_exp1.py \
    --lr 1e-5 --epochs 30 --batch_size 8 \
    --data_root data/ --splits_dir splits/ \
    --output_dir outputs/exp1/sweep_lr1e5

---
## 6. Inspect Sweep Results and Choose Best LR

We inspect the saved `history.json` files and select the LR with the
highest validation mIoU on ACD1K.

In [ ]:
# Cell 12 — Compare sweep histories
import json

def best_miou(path):
    with open(path) as f:
        hist = json.load(f)
    row = max(hist, key=lambda x: x['val_mIoU'])
    return row['epoch'], row['val_mIoU'], row['val_F1'], row['val_MAE']

runs = {
    'lr=1e-4': 'outputs/exp1/sweep_lr1e4/final/history.json',
    'lr=6e-5': 'outputs/exp1/sweep_lr6e5/final/history.json',
    'lr=1e-5': 'outputs/exp1/sweep_lr1e5/final/history.json',
}

for label, path in runs.items():
    try:
        ep, miou, f1, mae = best_miou(path)
        print(f"{label}: best epoch={ep}, val mIoU={miou:.4f}, F1={f1:.4f}, MAE={mae:.4f}")
    except FileNotFoundError:
        print(f"{label}: history.json not found at {path}")

> After inspecting the printed results above, choose the best learning
> rate (1e-4) and use it for the final training run below.

---
## 7. Final Training Run (best LR)

We now run a longer training with early stopping using the best learning
rate from the sweep. This run writes its artifacts to
`outputs/exp1/final/` and is the only checkpoint consumed by the
project's unified final evaluation.

In [ ]:
# Cell 13 — Final Experiment 1 training (update --lr if sweep differs)
!PYTHONPATH=/content/AI-final-project/src \
 python src/engine_exp1.py \
    --lr 1e-4 --epochs 60 --batch_size 8 \
    --data_root data/ --splits_dir splits/ \
    --output_dir outputs/exp1

---
## 8. Verify Outputs and Handoff to Final Evaluation

Experiment 1's responsibility ends with producing the best validation
checkpoint and training history. The unified 200-image hold-out
evaluation is handled elsewhere.

In [ ]:
# Cell 14 — Confirm best_model.pth and history.json
import os

paths = [
    'outputs/exp1/final/best_model.pth',
    'outputs/exp1/final/history.json',
]
for p in paths:
    print('✅' if os.path.exists(p) else '❌ NOT FOUND', p)

If both files above exist and look reasonable, Experiment 1 is complete.

- `outputs/exp1/final/best_model.pth` — SINetV2 weights + metadata  
- `outputs/exp1/final/history.json` — per-epoch train/val metrics

These artifacts can now be passed to the project's final evaluation
notebook for unified testing on the 200-image hold-out set.

---
## 9. Sync Notebook + Exp1 Outputs to GitHub

Run this section after training completes if you want to push the latest
`notebooks/02_train_exp1_yansong.ipynb` and all `outputs/exp1/` artifacts
back to the GitHub repository.

In [ ]:
# Cell 15 — Commit and push Exp1 notebook + outputs to GitHub
# Note: In Colab, you need authentication (PAT) to push.

import os

# Optional: configure identity for this Colab runtime only
!git config user.email "ys.jia@outlook.com"
!git config user.name "y4medi"

# Ensure we are on the Yansong branch
!git checkout Yansong

# Add updated Exp1 notebook and all Exp1 outputs
!git add notebooks/02_train_exp1_yansong.ipynb
!git add outputs/exp1/

# Commit if there are staged changes
status = !git status --porcelain
if len(status) == 0:
    print("No changes to commit.")
else:
    !git commit -m "Exp1: update SINetV2 notebook and outputs"

# Push (Colab may prompt for credentials/token)
!git push -u origin Yansong

# Optional: download final artifacts locally
from google.colab import files
if os.path.exists('outputs/exp1/final/best_model.pth'):
    files.download('outputs/exp1/final/best_model.pth')
if os.path.exists('outputs/exp1/final/history.json'):
    files.download('outputs/exp1/final/history.json')